Lấy tất cả bài viết trong ngày

In [1]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Cấu hình

In [2]:
today = date.today().strftime("%Y-%m-%d")
headers = {"User-Agent": "Mozilla/5.0"}

all_links = []
page = 1

# 2. Lấy danh sách link

In [9]:
print(f"=== LINK NGÀY {today} ===")
while True:
    if page == 1:
        url = f"https://dantri.com.vn/thoi-su/from/{today}/to/{today}.htm"
    else:
        url = f"https://dantri.com.vn/thoi-su/from/{today}/to/{today-1}/trang-{page}.htm"

    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = soup.find_all(
        "article",
        class_="article-item",
        attrs={"data-content-name": f"category-timeline_page_{page}"}
    )

    if not articles:
        break

    for art in articles:
        link = art.get("data-content-target")
        if link:
            if link.startswith("/"):
                link = "https://dantri.com.vn" + link
                all_links.append(link)
    page += 1
print("\nTỔNG LINK:", len(all_links))
for l in all_links:
    print(l)


=== LINK NGÀY 2026-01-29 ===

TỔNG LINK: 27
https://dantri.com.vn/thoi-su/thu-tuong-pham-minh-chinh-hoi-kien-chu-tich-hoi-dong-chau-au-antonio-costa-20260129144840130.htm
https://dantri.com.vn/thoi-su/kiem-soat-chat-chat-luong-thuc-pham-ngay-tet-xu-ly-nghiem-vi-pham-20260129144730188.htm
https://dantri.com.vn/thoi-su/xe-bon-xa-thai-keo-dai-5m-trong-luc-di-chuyen-tren-duong-o-tphcm-20260129140009247.htm
https://dantri.com.vn/thoi-su/ho-con-rua-phu-sac-xanh-ngoc-sau-mot-tuan-chinh-trang-20260129141514473.htm
https://dantri.com.vn/thoi-su/30-o-to-ngap-nuoc-duoc-dau-gia-truc-tuyen-voi-khoi-diem-68-ty-dong-20260129140333917.htm
https://dantri.com.vn/thoi-su/dua-nghi-quyet-thanh-hanh-dong-mang-luong-sinh-khi-moi-cho-cong-dong-quoc-te-20260129105630529.htm
https://dantri.com.vn/thoi-su/nhung-diem-dac-biet-net-moi-cua-cuoc-thi-sang-kien-an-toan-giao-thong-2025-20260129091626301.htm
https://dantri.com.vn/thoi-su/truy-tim-nguoi-dan-ong-ra-giua-duong-hat-karaoke-o-dong-nai-20260129125531310.htm
h

# 3. Hàm crawl

In [4]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    # ===== TITLE =====
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # ===== BODY =====
    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "content": body
    }

In [5]:
print("\n=== TEST 1 BÀI ===")
test_url = all_links[2]
data = crawl_article(test_url)

print("URL:", data["url"])
print("TITLE:", data["title"])
print("CONTENT:")
print(data["content"])


=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/xe-bon-xa-thai-keo-dai-5m-trong-luc-di-chuyen-tren-duong-o-tphcm-20260129140009247.htm
TITLE: Xe bồn xả thải kéo dài 5m trong lúc di chuyển trên đường ở TPHCM
CONTENT:
Ngày 29/1, người dùng mạng xã hội lan truyền clip ghi lại cảnh một xe bồn trộn bê tông xả chất thải trong lúc lưu thông trên đường.
Qua xác minh, vụ việc xảy ra vào chiều 26/1 trên đường Đặng Công Bỉnh, đoạn gần giao lộ Đặng Công Bỉnh - Nguyễn Văn Bứa, thuộc xã Bà Điểm, TPHCM.
Xe bồn trộn bê tông xả thải trên đường ở xã Bà Điểm (Ảnh: Cắt từ clip).
Hình ảnh từ clip cho thấy, xe bồn trộn bê tông mang biển số 62C-100.xx đang di chuyển, bất ngờ để nước thải từ bồn chứa chảy tràn xuống mặt đường, kéo dài hơn 5m. Thời điểm này, nhiều học sinh đi bộ và người điều khiển xe máy lưu thông qua khu vực suýt bị nước bẩn văng trúng người. Toàn bộ sự việc được camera hành trình của một phương tiện phía sau ghi lại.
Liên quan đến vụ việc, Đội CSGT An Lạc thuộc Phòng CSGT (PC08, Công

# 4. Cấu hình Splitter

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",   # đoạn
        "\n",     # dòng
        ".", "!", "?", ":", ";", ","
    ]
)
dulieu = splitter.split_text(data["content"])
print("Số chunk:", len(dulieu))
for i, chunk in enumerate(dulieu):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)


Số chunk: 2

--- CHUNK 0 ---
Ngày 29/1, người dùng mạng xã hội lan truyền clip ghi lại cảnh một xe bồn trộn bê tông xả chất thải trong lúc lưu thông trên đường.
Qua xác minh, vụ việc xảy ra vào chiều 26/1 trên đường Đặng Công Bỉnh, đoạn gần giao lộ Đặng Công Bỉnh - Nguyễn Văn Bứa, thuộc xã Bà Điểm, TPHCM.
Xe bồn trộn bê tông xả thải trên đường ở xã Bà Điểm (Ảnh: Cắt từ clip).
Hình ảnh từ clip cho thấy, xe bồn trộn bê tông mang biển số 62C-100.xx đang di chuyển, bất ngờ để nước thải từ bồn chứa chảy tràn xuống mặt đường, kéo dài hơn 5m. Thời điểm này, nhiều học sinh đi bộ và người điều khiển xe máy lưu thông qua khu vực suýt bị nước bẩn văng trúng người. Toàn bộ sự việc được camera hành trình của một phương tiện phía sau ghi lại.

--- CHUNK 1 ---
Liên quan đến vụ việc, Đội CSGT An Lạc thuộc Phòng CSGT (PC08, Công an TPHCM) cho biết, đơn vị đã nắm thông tin và đang xác minh, mời tài xế xe bồn lên làm việc để xử lý theo quy định.
Thông tin doanh nghiệp - sản phẩm


# 5. Xử lý hàng loạt

In [7]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/27] Đang xử lý: https://dantri.com.vn/thoi-su/thu-tuong-pham-minh-chinh-hoi-kien-chu-tich-hoi-dong-chau-au-antonio-costa-20260129144840130.htm
  -> OK: 4 chunks
[2/27] Đang xử lý: https://dantri.com.vn/thoi-su/kiem-soat-chat-chat-luong-thuc-pham-ngay-tet-xu-ly-nghiem-vi-pham-20260129144730188.htm
  -> OK: 5 chunks
[3/27] Đang xử lý: https://dantri.com.vn/thoi-su/xe-bon-xa-thai-keo-dai-5m-trong-luc-di-chuyen-tren-duong-o-tphcm-20260129140009247.htm
  -> OK: 2 chunks
[4/27] Đang xử lý: https://dantri.com.vn/thoi-su/ho-con-rua-phu-sac-xanh-ngoc-sau-mot-tuan-chinh-trang-20260129141514473.htm
  -> OK: 4 chunks
[5/27] Đang xử lý: https://dantri.com.vn/thoi-su/30-o-to-ngap-nuoc-duoc-dau-gia-truc-tuyen-voi-khoi-diem-68-ty-dong-20260129140333917.htm
  -> OK: 3 chunks
[6/27] Đang xử lý: https://dantri.com.vn/thoi-su/dua-nghi-quyet-thanh-hanh-dong-mang-luong-sinh-khi-moi-cho-cong-dong-quoc-te-20260129105630529.htm
  -> OK: 10 chunks
[7/27] Đang xử lý: h

In [8]:
# 6. Lưu file JSON
output_file = "dulieu.json"
result = {
    "date": today,
    "articles": final_data
} 
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)
print(f"Đã lưu dữ liệu vào file: {output_file}")

Đã lưu dữ liệu vào file: dulieu.json
